<a href="https://colab.research.google.com/github/vincemutua/academic-rag-assistant/blob/main/book_knowledge_rag_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation

In [1]:
# Cloning the github repository
!git clone https://github.com/vincemutua/academic-rag-assistant.git

#Moving to working directory
%cd academic-rag-assistant

fatal: destination path 'academic-rag-assistant' already exists and is not an empty directory.
/content/academic-rag-assistant


### Installing dependencies

In [2]:
!pip install -U -q \
  langchain \
  langchain-core \
  langchain-classic \
  langchain-google-genai \
  langchain-huggingface \
  langchain-text-splitters \
  sentence-transformers \
  faiss-cpu \
  pypdf

In [3]:
# Install LangChain, FAISS, and HuggingFace components
!pip install -q langchain langchain-community sentence-transformers faiss-cpu pypdf google-generativeai

### Document Ingestion and Chunking

In [4]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("1. Loading textbooks...")
loader = PyPDFDirectoryLoader("docs/")
pages = loader.load()

print("2. Chunking text...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

print("3. Building FAISS vector database (Downloading model)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("\nDatabase built and ready to search!")

/tmp/ipykernel_15612/1042875707.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


1. Loading textbooks...
2. Chunking text...
3. Building FAISS vector database (Downloading model)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Database built and ready to search!


In [5]:
import os
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API Key: ")

# Using the required 3.6-flash model
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

Enter your Gemini API Key: ··········


In [7]:
# The context stays in the system prompt, but the {input} moves to the human prompt!
personalized_template = (
    "You are an expert programming tutor. Explain the student's topic using the provided textbook context. \n"
    "However, to make it easier to understand, you MUST explain the concept using an analogy "
    "based on tactical football squad management and midfield positioning. \n"
    "Explain how the programming concept relates to player roles, independent movement, or formations. \n\n"
    "Context:\n{context}"
)

personalized_prompt = ChatPromptTemplate.from_messages([
    ("system", personalized_template),
    ("human", "{input}"), # <- This was missing!
])

personalized_chain = create_retrieval_chain(
    retriever,
    create_stuff_documents_chain(llm, personalized_prompt)
)

# Test the personalized explanation
question = "Explain the concept of Orthogonality."
print(f"Student: {question}\n")
print("Tutor is thinking...\n")

response = personalized_chain.invoke({"input": question})
print(response["answer"])

Student: Explain the concept of Orthogonality.

Tutor is thinking...



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### What is Orthogonality?

In programming, **orthogonality** refers to **independence or decoupling**. Two components are orthogonal if a change in one does not affect the other. 

In geometry, two lines are orthogonal if they meet at right angles (like the X and Y axes on a graph). If you move along the X-axis, your position on the Y-axis remains completely unchanged. In software engineering, this means if you change your user interface (UI), your database code shouldn't break, and vice versa.

---

### The Tactical Football Analogy: Midfield Positioning and Role Independence

To understand orthogonality, imagine managing a high-performing football squad—specifically setting up your midfield engine room.

#### 1. Orthogonal Midfield Roles (A Well-Designed System)
Imagine a midfield trio with clearly defined, independent responsibilities:
* **The Defensive Midfielder (The Database):** Their job is to break up play and hold the central zone laterally. 
* **The Box-to-Box Midfielder (Bu

In [8]:
planner_template = (
    "You are an academic strategist. Based ONLY on the provided textbook context, "
    "create a 3-day revision plan utilizing active recall for the student's topic. \n"
    "Include a brief summary, 3 challenging active recall questions, and cite the exact page numbers.\n\n"
    "Context:\n{context}"
)

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", planner_template),
    ("human", "{input}")
])

planner_chain = create_retrieval_chain(
    retriever,
    create_stuff_documents_chain(llm, planner_prompt)
)

print("Generating plan...\n")
plan = planner_chain.invoke({"input": "The DRY Principle"})
print(plan["answer"])

Generating plan...



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### **Summary of the Topic**
The DRY principle stands for **"Don't Repeat Yourself."** It dictates that *"Every piece of knowledge must have a single, unambiguous, authoritative representation within a system."* 

In software development, maintenance is not a isolated activity but a routine, continuous part of the entire process driven by changing environments, evolving requirements, and shifting developer understanding. Duplicating knowledge—whether in specifications, processes, or programs—creates severe maintenance problems that begin long before an application ships. Following the DRY principle ensures software is developed reliably and remains easier to understand and maintain.

---

### **3-Day Active Recall Revision Plan**

#### **Day 1: Core Principles & Definitions**
* **Focus:** Master the definition of the DRY principle and the nature of software maintenance.
* **Method:** Read the context once, close it, and attempt to write down the exact definition of DRY from memory. 
* 

In [9]:
!pip freeze | grep -E "langchain|faiss|sentence-transformers|google-genai" > requirements.txt